In [2]:
import csv, time, json, ast
from seleniumbase import Driver
from pprint import pprint
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import os
import shutil
import csv

In [3]:
website = 'IND Distribution'
browser = Driver(uc=True, incognito=True)
browser_wait = WebDriverWait(browser, 60)
# browser.maximize_window()

In [4]:
browser.get('https://ind-distribution.com/pages/shop-by-brand')

desired_brands = ['3D Design']

url_brands = []

for desired_brand in desired_brands:
    products_brand = browser.find_element(by=By.CSS_SELECTOR, value=f'.shop-by-brand [title="{desired_brand}"]').get_attribute('href')
    url_brands.append(products_brand)

print(url_brands)

['https://ind-distribution.com/collections/3d-design']


In [5]:
products_list = []
with open(f'3D Design-products.csv', 'a', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow(['vendor', 'link'])

    for link in url_brands:
        vendor = link.split('/collections/')[-1].replace('-',' ').title()
        proceed = True
        current_page = 1

        while(proceed):
            url_product_page = link + "?page=" + str(current_page)
            browser.get(url_product_page)

            products = browser.find_elements(by=By.CSS_SELECTOR, value='.collection__item .grid__image')
            for i in products:
                product = i.get_attribute('href')
                writer.writerow([vendor, product])
                products_list.append(product)
            products_list = list(set(products_list)) 
            print(len(products_list))

            product_existence = browser.find_elements(by=By.CSS_SELECTOR, value='.collection__item .grid__image')
            if product_existence == []:
                proceed = False
            else:
                current_page += 1

12
24
36
48
60
72
84
96
108
120
132
144
156
168
180
192
204
216
228
240
252
264
276
288
300
312
315
315


Scrape

In [ ]:
missed_sku = []
header = ['link','title', 'desc', 'images', 'check variants', 'option titles', 'options', 'vendor']
with open(f'3D Design-scrape.csv', 'a', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    # writer.writerow(header)

    df1 = pd.read_csv(r"3D Design-products.csv", dtype=str)
    for index, row in df1.iterrows():
        link = row['link']
        vendor = row['vendor']
        # browser.get(f'https://ind-distribution.com/collections/ind/products/ind-e46-m3-csl-s54-carbon-airbox-installation-kit')
        browser.get(f'{link}')
        time.sleep(2)
        try:
            title = browser.find_element(by=By.CSS_SELECTOR, value='[itemprop="name"]').get_attribute('innerText').strip()
            sku = browser.find_element(by=By.CSS_SELECTOR, value='[class="variant-sku"]').get_attribute('innerText').strip()
            desc = browser.find_element(by=By.CSS_SELECTOR, value='[class="product-single__description rte"]').get_attribute('innerHTML').strip()
            # price = browser.find_element(by=By.CSS_SELECTOR, value='[class="product-single__price"]').get_attribute('innerText').strip()

            images_list = []
            images = browser.find_elements(by=By.CSS_SELECTOR, value='[class="grid--full"] .product-single__thumbnails .grid__item img')
            for imgi in images:
                image = imgi.get_attribute('src')
                images_list.append(image)

            check_variants = browser.find_element(by=By.CSS_SELECTOR, value='.product-select-title').get_attribute('style')
            if "display: none" in check_variants:
                is_variant = False
            else:
                is_variant = True

            option_titles = []
            variant_titles = browser.find_elements(by=By.CSS_SELECTOR, value='#AddToCartForm .selector-wrapper label')
            for i in variant_titles:
                variant_title = i.get_attribute('innerText')
                option_titles.append(variant_title)

            ###make option list check to replace "/" in option where "/" is not option separator
            option_refer = {}
            option_checks = browser.find_elements(by=By.CSS_SELECTOR, value='.single-option-selector option')
            for i in option_checks:
                option_chk = i.get_attribute('innerText').strip()
                option_rp = option_chk.replace(' / ', ' or ').strip()
                option_refer[option_chk] = option_rp

            # print(option_refer)

            variations = browser.find_elements(by=By.CSS_SELECTOR, value='#productSelect option')
            options = []
            for variant in variations:
                option_values = {}
                variant_sku = variant.get_attribute('data-sku').strip()
                option = variant.get_attribute('innerText').strip()
                price = option.split(' - $')[-1]
                
                # print(option)
                for key, value in option_refer.items():
                    option = option.replace(key, value)
 
                # print(option)
                option0 = option.split(' - $')[0]
                # print(option0)

                count = option0.lower().count(" / ")
                # print(count)
                if count == 0:
                    option1 = option0
                    option_values['option1'] = option1
                elif count == 1:
                    option1 = option0.split(' / ')[0]
                    option2 = option0.split(' / ')[-1]
                    option_values['option1'] = option1
                    option_values['option2'] = option2
                elif count == 2:
                    option1 = option0.split(' / ')[0]
                    option2 = option0.split(' / ')[1]
                    option3 = option0.split(' / ')[-1]
                    option_values['option1'] = option1
                    option_values['option2'] = option2
                    option_values['option3'] = option3
                
                option_values['sku'] = variant_sku
                option_values['price'] = price
                options.append(option_values)

            writer.writerow([link, title, desc, images_list, is_variant, option_titles, options, vendor])
            print([link, title, desc, images_list, is_variant, option_titles, options, vendor])
            # break
        except Exception as e:
            print(f"{link}: {e}")
            missed_sku.append(link)

print(missed_sku)

['https://ind-distribution.com/collections/3d-design/products/3d-design-a90-supra-carbon-front-lip', '3D Design A90 Supra Carbon Front Lip', "<p>3D Design is renowned for their line of BMW aero parts and accessories. From inception, 3D Design has worked diligently to expand their product line and with great success has created a wide range of parts and accessories for enthusiasts. 2012 marked the expansion of their lifestyle accessories and interior components as well as the introduction of suspension to the export markets. 3D Design utilizes only the best materials, manufacturers, and designers to create products that are in harmony with the intentions of Toyota.</p>\n<p>3D Design chooses not to focus on time and cost, but instead concentrates on detail and accuracy. From the tight fitment of the exterior components to the flawless carbon weave in their CFRP components, 3D Design's detail oriented philosophy can be seen in every piece available for the Toyota MK5 Supra.</p>\n<p>IND Di

HTML

In [3]:
csv_path = r"3D Design-scrape.csv"
scrape_df = pd.read_csv(csv_path).fillna('')

final_data = []

for index0, row in scrape_df.iterrows():

    url = row['link']
    # print(url)
    title = row['title']
    desc = row['desc']
    vendor = row['vendor']
    check_variants = row['check variants']

    images = ast.literal_eval(row['images'])

    handle = (re.sub(r'[^a-zA-Z0-9\n\.]', '-', title).replace(".", "-").replace("---", "-").replace("--", "-")).lower()
    if handle.endswith('-'):
        handle = handle[:-1]

    option_titles = ast.literal_eval(row['option titles'].replace('*',''))
    variations = ast.literal_eval(row['options'])

    part_no = ''
    if check_variants == False:
        for i in variations:
            part_no += f"<p>{i['sku']}</p>"
    else:
        for i in variations:
            part_entry = f"<p>{i['sku']} ({i['option1']}"
            
            if 'option2' in i:
                part_entry += f" / {i['option2']}"
            
            if 'option3' in i:
                part_entry += f" / {i['option3']}"

            part_entry += ")</p>"
            part_no += part_entry

    #######images & variations comparison#######
    len_image = len(images)
    len_variant = len(variations)

    diff = abs(len_image - len_variant)

    if len_image > len_variant:
        for _ in range(diff):
            variations.append({key: '' for key in variations[0].keys()})
        option_titles.extend(['']*diff)
    elif len_variant > len_image:
        images.extend(['']*diff)
        option_titles.extend(['']*diff)
    # print(option_titles)

    while len(option_titles) < 3:
        option_titles.append('')

    option_title1 = option_titles[0]
    option_title2 = option_titles[1]
    option_title3 = option_titles[2]

    def description(desc, part_no, vendor):
        return f"""<h4><strong>Description</strong></h4>{desc}
        <p>&nbsp;</p>
        <h4>Compatibility</h4><p>Feel free to contact us at info@mlperformance.co.uk should you wish to double check!</p>
        <p>&nbsp;</p>
        <h4>Compatibility Check</h4><p>To ensure the part(s) you have ordered fits your vehicle, we run a compatibility check prior to dispatch. We can do this either using your registration number (UK) or the last 7 digits of your VIN. Simply enter your car details prior to checkout.</p>
        <p>&nbsp;</p>
        <h4>Part Number</h4>{part_no}
        <p>&nbsp;</p>
        <h4>More Information</h4><p><strong>Manufactured by</strong></p><p>{vendor}</p>"""

    # Loop through images
    for index, image in enumerate(images, start=0):

        variant_sku = variations[index]['sku']
        option_value1 = variations[index]['option1']
        try:
            option_value2 = variations[index]['option2']
        except:
            option_value2 = ''
        try:
            option_value3 = variations[index]['option3']
        except:
            option_value3 = ''

        price = variations[index]['price']
        # Create a new dictionary for each image
        info = {}
        
        info['Handle'] = handle
        info['Title'] = title
        info['Body (HTML)'] = description(desc, part_no, vendor).replace("\n", "")
        info['Vendor'] = vendor
        info['Standardized Product Type'] = None
        info['Custom Product Type'] = None
        info['Tags'] = f"Uploaded by_Muazzim, Brand_{vendor}, Product Type_"
        info['Published'] = "TRUE"
        info['Option1 Name'] = option_title1
        info['Option1 Value'] = option_value1
        info['Option2 Name'] = option_title2
        info['Option2 Value'] = option_value2
        info['Option3 Name'] = option_title3
        info['Option3 Value'] = option_value3

        info['Variant SKU'] = variant_sku
        info['Variant Grams'] = None
        info['Variant Inventory Tracker'] = "shopify"
        info['Variant Inventory Policy'] = 'continue'
        info['Variant Fulfillment Service'] = 'manual'
        info['Variant Price'] = None
        info['Variant Compare At Price'] = price
        info['Variant Requires Shipping'] = 'TRUE'
        info['Variant Taxable'] = 'TRUE'
        info['Variant Barcode'] = None

        # For each image, create a new entry
        info['Image Src'] = image
        info['Image Position'] = index + 1
        info['Image Alt Text'] = title
        info['Gift Card'] = None
        info['SEO Title'] = title
        info['SEO Description'] = 'Get ' + title + ' for your car to get your desired looks and performance from ML Performance at the lowest price with FREE UK shipping & next day delivery on in stock items. Very cheap prices & good service.'
        info['Google Shopping / Google Product Category'] = None
        info['Google Shopping / Gender'] = None
        info['Google Shopping / Age Group'] = None
        info['Google Shopping / MPN'] = variant_sku
        info['Google Shopping / AdWords Grouping'] = None
        info['Google Shopping / AdWords Labels'] = None
        info['Google Shopping / Condition'] = 'new'
        info['Google Shopping / Custom Product'] = None
        info['Google Shopping / Custom Label 0'] = None
        info['Google Shopping / Custom Label 1'] = None
        info['Google Shopping / Custom Label 2'] = None
        info['Google Shopping / Custom Label 3'] = None
        info['Google Shopping / Custom Label 4'] = None
        info['Variant Image'] = None
        info['Variant Weight Unit'] = 'kg'
        info['Variant Tax Code'] = 8708949900
        info['Cost per item'] = None
        info['Margins'] = None
        info['Price / International'] = None
        info['Compare At Price / International'] = None
        info['Status'] = 'active'

        # Append the new dictionary to the final_data list
        final_data.append(info)

final_df = pd.DataFrame(final_data)

output_path = os.path.join("3D Design-HTML.csv")
final_df.to_csv(output_path, index=False)

print('File saved and moved to desired folder')


File saved and moved to desired folder
